# Evaluate Document Chat on a sentiment-analysis paper

This notebook evaluates the app pipeline with 24 manually written, page-labelled questions. It measures retrieval (hit@6 and MRR@6) and whether structured answers cite the labelled page. Reference answers are included for human review; a matching citation alone is not a correctness score.

Install `requirements.txt` in the notebook Python environment and ensure local Ollama has `llama3.2` and `nomic-embed-text`. The PDF remains at its local path below; change `PDF_PATH` to use a different copy. Model responses can vary across runs.


In [ ]:
import json
import os
import re
import sys
from pathlib import Path

PROJECT_DIR = Path(r'C:\Users\lolen\Downloads\Jupyter Mac\Python\2-AI\LLM\LLM-Repo\Projects\RAG-Document-Chat')
PDF_PATH = Path(r'C:\Users\lolen\Desktop\RWTH\WS2026\My Notes\Academic\Thesis\I5 Chair\Seminar\Challenges in LLM-Based Sentiment Analysis_7.pdf')
CASES_PATH = PROJECT_DIR / 'eval_cases.sentiment.json'

# Optional for a temporary evaluation environment; a normal project install needs nothing here.
if os.environ.get('RAG_EVAL_DEPS'):
    sys.path.insert(0, os.environ['RAG_EVAL_DEPS'])
sys.path.insert(0, str(PROJECT_DIR))
from pypdf import PdfReader
from document_chat import ABSTAIN, RagIndex, extract_pdfs

print('Project:', PROJECT_DIR)
print('PDF:', PDF_PATH)


In [ ]:
dataset = json.loads(CASES_PATH.read_text(encoding='utf-8'))
cases = dataset['cases']
pages = len(PdfReader(str(PDF_PATH)).pages)
assert cases and len({case['id'] for case in cases}) == len(cases)
assert all(case['source'] == PDF_PATH.name and 1 <= case['page'] <= pages for case in cases)
chunks, warnings = extract_pdfs([(PDF_PATH.name, PDF_PATH.read_bytes())])
print(f'{len(cases)} cases, {pages} PDF pages, {len(chunks)} indexed passages')
print('Warnings:', warnings)


In [ ]:
index = RagIndex.build(chunks)
print('Built in-memory Chroma index with local Ollama embeddings.')


In [ ]:
METHODS = ('hybrid', 'semantic', 'keyword')
retrieval_rows = []
for method in METHODS:
    for case in cases:
        hits = index.search(case['question'], method)
        rank = next((i for i, hit in enumerate(hits, 1)
                     if hit.chunk.source == case['source'] and hit.chunk.page == case['page']), None)
        retrieval_rows.append({'id': case['id'], 'method': method, 'rank': rank})

def retrieval_summary(rows, method):
    subset = [row for row in rows if row['method'] == method]
    n = len(subset)
    return {'method': method, 'cases': n,
            'hit_at_6': round(sum(row['rank'] is not None for row in subset) / n, 3),
            'mrr_at_6': round(sum(1 / row['rank'] if row['rank'] else 0 for row in subset) / n, 3)}

for method in METHODS:
    print(json.dumps(retrieval_summary(retrieval_rows, method)))
print('Hybrid misses:', [row['id'] for row in retrieval_rows
                         if row['method'] == 'hybrid' and row['rank'] is None])


In [ ]:
# This runs the complete retrieval -> context -> structured Ollama answer path.
answer_rows = []
try:
    for position, case in enumerate(cases, 1):
        try:
            answer, context = index.answer(case['question'], 'hybrid')
            citations = [int(value) for value in re.findall(r'\[(\d+)\]', answer)]
            cited_gold = any(1 <= number <= len(context)
                             and context[number - 1].chunk.source == case['source']
                             and context[number - 1].chunk.page == case['page']
                             for number in citations)
            row = {'id': case['id'], 'category': case['category'], 'question': case['question'],
                   'gold_page': case['page'], 'expected_answer': case['expected_answer'],
                   'answer': answer, 'answered': answer != ABSTAIN,
                   'citations': citations, 'cited_gold_page': cited_gold,
                   'context_pages': [hit.chunk.page for hit in context], 'error': None}
        except Exception as exc:
            row = {'id': case['id'], 'category': case['category'], 'question': case['question'],
                   'gold_page': case['page'], 'expected_answer': case['expected_answer'],
                   'answer': '', 'answered': False, 'citations': [], 'cited_gold_page': False,
                   'context_pages': [], 'error': f'{type(exc).__name__}: {exc}'}
        answer_rows.append(row)
        print(f"{position:02d}/{len(cases)} {row['id']} answered={row['answered']} cited_gold={row['cited_gold_page']} error={row['error']}")
finally:
    index.close()

valid_rows = [row for row in answer_rows if row['error'] is None]
summary = {'cases': len(answer_rows), 'errors': len(answer_rows) - len(valid_rows),
           'answered': sum(row['answered'] for row in valid_rows),
           'cited_gold_page': sum(row['cited_gold_page'] for row in valid_rows),
           'abstained': sum(not row['answered'] for row in valid_rows)}
print('Answer summary:', json.dumps(summary))


In [ ]:
# Review answers against the human-written references. Citation checks do not prove correctness.
for row in answer_rows:
    print(f"\n{row['id']} | gold PDF page {row['gold_page']} | context pages {row['context_pages']}")
    print('Question: ', row['question'])
    print('Expected: ', row['expected_answer'])
    print('Observed: ', row['answer'] or row['error'])


## Reading the results

- `hit@6` asks whether the labelled page appears among the top six retrieved passages; `MRR@6` rewards a higher rank.
- `cited_gold_page` checks the citation number in the displayed answer against the labelled page. It cannot establish whether the wording is factually correct.
- Compare each observed answer with its reference in the final cell. Some questions have relevant information on adjacent pages, so a missing labelled page is a strict retrieval failure rather than automatic proof that the response is wrong.
